<div style="direction: rtl; white-space: normal; line-height: 1.8;">
<h1>NB12g — آزمایش بازسازی کیفیت Golden Dataset</h1>
<p>
این نوت‌بوک جایگزین اجرایی زنجیرهٔ پراکندهٔ 12 تا 12f است؛ اما هیچ‌کدام از
فایل‌های قدیمی را حذف یا بازنویسی نمی‌کند. ورودی‌های آن artifactهای ثابت‌شدهٔ
NB10/NB11، Golden Dataset v1 و provenance بازبینی انسانی است.
</p>
<p>
مشکل نسخهٔ اول صرفاً کمبود مدل embedding نبود. در NB12e سؤال‌های انسانی چند بار
با templateهای عمومی و کلمه‌هایی که به‌صورت چرخشی از پاسخ انتخاب شده بودند
جایگزین شدند. به همین دلیل بازیابی با خود evidence قوی، ولی بازیابی با سؤال‌ها
بسیار ضعیف بود. این نوت‌بوک سؤال‌ها را از evidence واقعی و chunkهای منبع تولید
می‌کند و کیفیت را با gateهای مستقل کنترل می‌کند.
</p>
<h3>اصول غیرقابل‌مذاکره</h3>
<ul>
  <li>Golden Dataset v1 تغییر نمی‌کند.</li>
  <li>هیچ stateای از اجرای نوت‌بوک دیگر لازم نیست؛ Restart + Run All پشتیبانی می‌شود.</li>
  <li>مسیرها از ریشهٔ پروژه کشف می‌شوند و به نام نوت‌بوک وابسته نیستند.</li>
<li>خروجی اولیه فقط یک artifact آزمایشی غیر‌فعال است و هیچ نسخهٔ جدیدی از Golden Dataset نیست.</li>
  <li>تولید LLM، embedding و داوری ماشینی cache می‌شوند.</li>
  <li>split قبل از expansion و در سطح source seed ساخته می‌شود تا leakage رخ ندهد.</li>
  <li>عملکرد retrieval فقط گزارش می‌شود و معیار حذف سؤال نیست؛ در غیر این صورت benchmark دچار سوگیری می‌شود.</li>
</ul>
</div>


<div style="direction: rtl; line-height: 1.8;">
<h2>۱. تنظیمات و کشف ریشهٔ پروژه</h2>
<p>
مقدارهای این بخش تنها تنظیمات قابل‌تغییر هستند. اجرای عادی هیچ فایل active را
جایگزین نمی‌کند. این نوت‌بوک هیچ‌گاه v2 نمی‌سازد؛ v2 فقط بعد از بازبینی دکتر و در جریان جداگانه ساخته می‌شود.
</p>
</div>


In [1]:
# Code Cell 01
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
from urllib.parse import urlparse
import hashlib
import importlib.util
import json
import math
import os
import random
import re
import shutil
import time
import uuid

import numpy as np

SOURCE_DATASET_FILENAME = "golden_dataset_v1.jsonl"
EXPERIMENT_DATASET_STEM = "golden_dataset_rebuild_experiment"
QUESTIONS_PER_SEED = 4
GENERATION_PROMPT_VERSION = "nb12g_question_generation_v2"
JUDGE_PROMPT_VERSION = "nb12g_machine_judge_v1"
CHAT_MAX_ATTEMPTS = 3
RANDOM_SEED = 14050621

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

current_path = Path.cwd().resolve()
project_root = next(
    (
        candidate
        for candidate in (current_path, *current_path.parents)
        if (candidate / "notebooks" / "phase2").is_dir()
        and (candidate / "data" / "phase2").is_dir()
    ),
    None,
)

assert project_root is not None, (
    "Project root could not be discovered. Open this notebook from inside "
    "the RAG_finance repository."
)

notebooks_phase2 = project_root / "notebooks" / "phase2"
golden_directory = notebooks_phase2 / "golden_dataset"
source_dataset_path = golden_directory / SOURCE_DATASET_FILENAME

assert source_dataset_path.is_file(), (
    f"Source Golden Dataset is missing: {source_dataset_path}"
)

run_started_at = datetime.now(timezone.utc)
print("Project root:", project_root)
print("Source dataset:", source_dataset_path)
print("Active-dataset promotion: disabled by notebook design")
print("\nPROJECT CONFIGURATION: PASS")


Project root: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance
Source dataset: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_v1.jsonl
Active-dataset promotion: disabled by notebook design

PROJECT CONFIGURATION: PASS


In [2]:
# Code Cell 02
def sha256_file(path: Path, block_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def read_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))


def read_jsonl(path: Path):
    records = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as error:
                raise AssertionError(
                    f"Invalid JSONL at {path}:{line_number}: {error}"
                ) from error
            assert isinstance(record, dict), (
                f"Non-object JSONL record at {path}:{line_number}"
            )
            records.append(record)
    return records


def write_json_atomic(path: Path, value) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(value, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    temporary.replace(path)


def write_jsonl_atomic(path: Path, records) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
    temporary.replace(path)


def canonical_text(value: str) -> str:
    translation = str.maketrans({
        "ي": "ی", "ى": "ی", "ك": "ک", "ۀ": "ه", "ة": "ه",
        "ؤ": "و", "إ": "ا", "أ": "ا", "ٱ": "ا",
        "\u200c": " ", "\u200f": " ", "\u200e": " ",
    })
    text = str(value or "").translate(translation).lower()
    text = re.sub(r"[ـ]+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def text_hash(value: str) -> str:
    return hashlib.sha256(canonical_text(value).encode("utf-8")).hexdigest()


def tokens(value: str):
    return re.findall(r"[\w\u0600-\u06ff]+", canonical_text(value))


def extract_first_json_object(text: str):
    text = (text or "").strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
    decoder = json.JSONDecoder()
    for position, character in enumerate(text):
        if character != "{":
            continue
        try:
            value, _ = decoder.raw_decode(text[position:])
        except json.JSONDecodeError:
            continue
        if isinstance(value, dict):
            return value
    return None


print("Shared functions loaded.")
print("\nFOUNDATION UTILITIES: PASS")


Shared functions loaded.

FOUNDATION UTILITIES: PASS


<div style="direction: rtl; line-height: 1.8;">
<h2>۲. اعتبارسنجی artifactهای ثابت‌شدهٔ NB10/NB11</h2>
<p>
این مرحله manifest خود NB11 را می‌خواند و hash و اندازهٔ تمام artifactهای ثبت‌شده
را کنترل می‌کند. بنابراین notebook به حافظه یا ترتیب اجرای NB11 وابسته نیست.
</p>
</div>


In [3]:
# Code Cell 03
freeze_manifest_matches = sorted(
    project_root.rglob("evaluation_freeze_manifest_v1.json")
)
assert len(freeze_manifest_matches) == 1, (
    "Expected exactly one NB11 evaluation freeze manifest; found "
    f"{len(freeze_manifest_matches)}."
)

freeze_manifest_path = freeze_manifest_matches[0]
freeze_manifest = read_json(freeze_manifest_path)

assert freeze_manifest.get("status") == "FROZEN_FOR_EVALUATION", (
    "NB11 is not frozen for evaluation."
)

artifact_descriptors = freeze_manifest.get("artifacts", {})
assert artifact_descriptors, "NB11 manifest has no artifacts."

artifact_failures = []
for artifact_name, descriptor in artifact_descriptors.items():
    artifact_path = project_root / descriptor["path"]
    checks = {
        "exists": artifact_path.is_file(),
        "bytes": artifact_path.is_file()
        and artifact_path.stat().st_size == descriptor.get("bytes"),
        "sha256": artifact_path.is_file()
        and sha256_file(artifact_path) == descriptor.get("sha256"),
    }
    if not all(checks.values()):
        artifact_failures.append({
            "artifact": artifact_name,
            "path": str(artifact_path),
            "checks": checks,
        })

assert not artifact_failures, (
    "Frozen NB11 artifact validation failed:\n"
    + json.dumps(artifact_failures, indent=2)
)

parent_child_descriptor = artifact_descriptors["parent_child_children"]
parent_descriptor = artifact_descriptors["parent_child_parents"]
structure_medium_descriptor = artifact_descriptors["structure_aware_medium"]

parent_child_path = project_root / parent_child_descriptor["path"]
parent_path = project_root / parent_descriptor["path"]
structure_medium_path = project_root / structure_medium_descriptor["path"]

print("NB11 manifest:", freeze_manifest_path.relative_to(project_root))
print("Artifacts validated:", len(artifact_descriptors))
print("Retrieval candidate:", parent_child_descriptor["path"])
print("\nNB10/NB11 FROZEN INPUT VALIDATION: PASS")


NB11 manifest: reports/phase2/chunking/evaluation_freeze_manifest_v1.json
Artifacts validated: 15
Retrieval candidate: data/phase2/chunks/nb11_benchmark_v1/parent_child_children_v1.jsonl

NB10/NB11 FROZEN INPUT VALIDATION: PASS


<div style="direction: rtl; line-height: 1.8;">
<h2>۳. کالبدشکافی و تثبیت seedهای نسخهٔ اول</h2>
<p>
سؤال‌های v1 به‌عنوان دادهٔ آموزشی یا الگوی تولید استفاده نمی‌شوند. تنها evidence،
answer پذیرفته‌شده، taxonomy و lineage از آن بازیابی می‌شود. این جداسازی از انتقال
templateهای معیوب به آزمایش بازسازی منتقل نمی‌شود.
</p>
</div>


In [4]:
# Code Cell 04
source_records = read_jsonl(source_dataset_path)
source_dataset_sha256 = sha256_file(source_dataset_path)

required_source_fields = {
    "candidate_id", "question", "expected_answer", "source_seed_id",
    "source_lineage", "source_evidence", "taxonomy_topics",
    "competency_ids",
}
source_contract_failures = [
    record.get("candidate_id", f"row:{index}")
    for index, record in enumerate(source_records, start=1)
    if not required_source_fields.issubset(record)
]
assert not source_contract_failures, (
    f"v1 schema failures: {source_contract_failures[:10]}"
)

source_records_by_seed = defaultdict(list)
for record in source_records:
    source_records_by_seed[int(record["source_seed_id"])].append(record)

assert sorted(source_records_by_seed) == list(range(1, 29)), (
    "The source dataset must cover seeds 1 through 28."
)

seed_registry = {}
for seed_id, records in sorted(source_records_by_seed.items()):
    evidence_values = {
        canonical_text(record["source_evidence"]): record["source_evidence"]
        for record in records
        if str(record.get("source_evidence", "")).strip()
    }
    answer_values = {
        canonical_text(record["expected_answer"]): record["expected_answer"]
        for record in records
        if str(record.get("expected_answer", "")).strip()
    }
    lineages = {record["source_lineage"] for record in records}

    assert len(evidence_values) == 1, (
        f"Seed {seed_id} has {len(evidence_values)} evidence texts."
    )
    assert len(answer_values) == 1, (
        f"Seed {seed_id} has {len(answer_values)} accepted answers."
    )
    assert lineages == {f"pilot_{seed_id}"}, (
        f"Seed {seed_id} has inconsistent lineage: {lineages}"
    )

    seed_registry[seed_id] = {
        "source_seed_id": seed_id,
        "source_lineage": f"pilot_{seed_id}",
        "approved_evidence": next(iter(evidence_values.values())),
        "approved_answer": next(iter(answer_values.values())),
        "taxonomy_topics": sorted({
            topic for record in records for topic in record["taxonomy_topics"]
        }),
        "competency_ids": sorted({
            item for record in records for item in record["competency_ids"]
        }),
        "legacy_record_count": len(records),
        "legacy_questions": [record["question"] for record in records],
    }

print("Source records:", len(source_records))
print("Source SHA-256:", source_dataset_sha256)
print("Unique evidence seeds:", len(seed_registry))
print("Legacy questions retained for diagnostics only: True")
print("\nSOURCE SEED FREEZE: PASS")


Source records: 573
Source SHA-256: df18bd384261400986e2c3bcdeda0cffa084243fb99f073c51396dbe2d9f2729
Unique evidence seeds: 28
Legacy questions retained for diagnostics only: True

SOURCE SEED FREEZE: PASS


In [5]:
# Code Cell 05
legacy_generic_patterns = [
    r"^برای یک فرد آگاه مالی، درباره .+ چه نکته‌ای باید بررسی شود؟$",
    r"^در یک تصمیم مالی واقعی، درباره .+ چه چیزی اهمیت بیشتری دارد؟$",
    r"^از دیدگاه آموزش مالی، درباره .+ چه مفهومی باید درک شود؟$",
    r"^با توجه به تجربه مطرح‌شده در منبع، درباره .+ چه تصمیمی باید گرفته شود؟$",
    r"^در شرایط عملی زندگی، درباره .+ چه اشتباهی باید جلوگیری شود؟$",
    r"^برای تحلیل بهتر این موضوع، درباره .+ چه رابطه‌ای میان مفاهیم وجود دارد؟$",
]

legacy_generic_count = sum(
    any(re.fullmatch(pattern, canonical_text(question)) for pattern in legacy_generic_patterns)
    for seed in seed_registry.values()
    for question in seed["legacy_questions"]
)

legacy_questions = [
    question
    for seed in seed_registry.values()
    for question in seed["legacy_questions"]
]
legacy_unique_ratio = (
    len({canonical_text(question) for question in legacy_questions})
    / len(legacy_questions)
)

legacy_diagnostic = {
    "records": len(legacy_questions),
    "seeds": len(seed_registry),
    "questions_per_seed_mean": len(legacy_questions) / len(seed_registry),
    "canonical_unique_ratio": legacy_unique_ratio,
    "known_generic_template_matches": legacy_generic_count,
    "use_as_generation_input": False,
}

print("=== LEGACY QUESTION DIAGNOSTIC ===")
for key, value in legacy_diagnostic.items():
    print(f"- {key}: {value}")
print("\nLEGACY QUESTION QUARANTINE: PASS")


=== LEGACY QUESTION DIAGNOSTIC ===
- records: 573
- seeds: 28
- questions_per_seed_mean: 20.464285714285715
- canonical_unique_ratio: 1.0
- known_generic_template_matches: 218
- use_as_generation_input: False

LEGACY QUESTION QUARANTINE: PASS


<div style="direction: rtl; line-height: 1.8;">
<h2>۴. اتصال provenance واقعی به chunkهای NB11</h2>
<p>
برای ۲۲ pilot اول از registry بازبینی انسانی و برای pilotهای ۲۳ تا ۲۸ از source
packet استفاده می‌شود. label مثبت از کل سند به‌صورت کور انتخاب نمی‌شود؛ chunkهای
همان سند بر اساس هم‌پوشانی با evidence پذیرفته‌شده محدود می‌شوند.
</p>
</div>


In [6]:
# Code Cell 06
human_registry_matches = sorted(
    project_root.rglob("human_review_registry.json")
)
human_registry_matches = [
    path for path in human_registry_matches
    if "before_" not in path.name
]
assert human_registry_matches, "human_review_registry.json was not found."
human_registry_path = max(
    human_registry_matches,
    key=lambda path: path.stat().st_mtime,
)

source_packet_matches = sorted(
    project_root.rglob("source_packet_pilots23_28_v1.json")
)
assert len(source_packet_matches) == 1, (
    "Expected one source_packet_pilots23_28_v1.json; found "
    f"{len(source_packet_matches)}."
)
source_packet_path = source_packet_matches[0]

human_registry = read_json(human_registry_path)
source_packet = read_json(source_packet_path)

completed_cases = human_registry.get("completed_human_review_cases", [])
author_cases = human_registry.get("author_review", {}).get(
    "cases_ready_for_review", []
)
source_packet_pilots = source_packet.get("pilots", [])

assert len(completed_cases) == 22, (
    f"Expected 22 completed human-review cases; found {len(completed_cases)}."
)
assert len(author_cases) == 28, (
    f"Expected 28 author-review cases; found {len(author_cases)}."
)
assert len(source_packet_pilots) == 6, (
    f"Expected 6 source-packet pilots; found {len(source_packet_pilots)}."
)

provenance_registry = {}

for seed_id, review_case in enumerate(completed_cases, start=1):
    document_ids = sorted({
        item.get("document_id")
        for item in review_case.get("source_evidence", [])
        if isinstance(item, dict) and item.get("document_id")
    })
    provenance_registry[seed_id] = {
        "pilot_case_id": review_case["pilot_case_id"],
        "provenance_source": "human_review_registry",
        "document_ids": document_ids,
        "review_case": review_case,
    }

packet_by_number = {
    int(pilot["reference_pilot_number"]): pilot
    for pilot in source_packet_pilots
}
assert sorted(packet_by_number) == list(range(23, 29))

for seed_id in range(23, 29):
    pilot = packet_by_number[seed_id]
    claim_document_ids = {
        claim.get("content_source", {}).get("document_id")
        for claim in pilot.get("claims", [])
        if isinstance(claim.get("content_source"), dict)
    }
    document_ids = sorted({
        value
        for value in (
            set(pilot.get("accepted_website_document_ids", []))
            | claim_document_ids
        )
        if isinstance(value, str) and value
    })
    provenance_registry[seed_id] = {
        "pilot_case_id": pilot["pilot_case_id"],
        "provenance_source": "source_packet",
        "document_ids": document_ids,
        "review_case": pilot,
    }

assert sorted(provenance_registry) == list(range(1, 29))

print("Human registry:", human_registry_path.relative_to(project_root))
print("Source packet:", source_packet_path.relative_to(project_root))
print("Provenance records:", len(provenance_registry))
print("Seeds without document IDs:", [
    seed_id for seed_id, item in provenance_registry.items()
    if not item["document_ids"]
])
print("\nPROVENANCE CONTRACT: PASS")


Human registry: data/phase2/human_review/human_review_registry.json
Source packet: data/phase2/human_review/source_packet_pilots23_28_v1.json
Provenance records: 28
Seeds without document IDs: [16]

PROVENANCE CONTRACT: PASS


In [7]:
# Code Cell 07
child_chunks = read_jsonl(parent_child_path)
parent_chunks = read_jsonl(parent_path)
structure_medium_chunks = read_jsonl(structure_medium_path)

assert len(child_chunks) == parent_child_descriptor["records"]
assert len(parent_chunks) == parent_descriptor["records"]
assert len(structure_medium_chunks) == structure_medium_descriptor["records"]

child_by_id = {chunk["chunk_id"]: chunk for chunk in child_chunks}
parent_by_id = {chunk["chunk_id"]: chunk for chunk in parent_chunks}

assert len(child_by_id) == len(child_chunks), "Duplicate child chunk IDs."
assert len(parent_by_id) == len(parent_chunks), "Duplicate parent chunk IDs."
assert all(
    chunk.get("parent_chunk_id") in parent_by_id
    for chunk in child_chunks
), "One or more child chunks have invalid parent references."

document_to_children = defaultdict(list)
for chunk in child_chunks:
    identifiers = set(chunk.get("member_document_ids", []))
    if chunk.get("source_id"):
        identifiers.add(chunk["source_id"])
    for identifier in identifiers:
        document_to_children[identifier].append(chunk)

stopwords = {
    "از", "به", "در", "با", "برای", "که", "این", "آن", "را", "و", "یا",
    "یک", "می", "شود", "شده", "است", "هست", "بود", "باشد", "های", "ها",
    "خود", "چه", "چرا", "چگونه", "درباره", "بر", "اگر", "اما", "هم",
}

def content_token_set(value: str):
    return {token for token in tokens(value) if len(token) > 2 and token not in stopwords}


def evidence_coverage(evidence: str, passage: str) -> float:
    evidence_tokens = content_token_set(evidence)
    if not evidence_tokens:
        return 0.0
    return len(evidence_tokens & content_token_set(passage)) / len(evidence_tokens)


evidence_linkage = {}
for seed_id, seed in seed_registry.items():
    provenance = provenance_registry[seed_id]
    candidate_chunks = []
    seen_ids = set()
    for document_id in provenance["document_ids"]:
        for chunk in document_to_children.get(document_id, []):
            if chunk["chunk_id"] not in seen_ids:
                candidate_chunks.append(chunk)
                seen_ids.add(chunk["chunk_id"])

    linkage_basis = "REGISTERED_DOCUMENT_ID"

    if not candidate_chunks:
        # Known transcript-ID gap (seed 16): recover only by exact, reviewed evidence.
        exact_evidence_units = [
            canonical_text(item.get("text", ""))
            for item in provenance.get("review_case", {}).get("source_evidence", [])
            if isinstance(item, dict)
            and len(canonical_text(item.get("text", ""))) >= 100
        ]
        for chunk in structure_medium_chunks:
            chunk_text = canonical_text(chunk["content"])
            if any(unit in chunk_text for unit in exact_evidence_units):
                candidate_chunks.append(chunk)
        linkage_basis = "EXACT_REVIEWED_EVIDENCE_FALLBACK"

    assert candidate_chunks, f"No provenance chunks were found for seed {seed_id}."

    scored = sorted(
        [
            (evidence_coverage(seed["approved_evidence"], chunk["content"]), chunk)
            for chunk in candidate_chunks
        ],
        key=lambda pair: (-pair[0], pair[1]["chunk_id"]),
    )
    best_score = scored[0][0]
    relative_floor = max(0.18, best_score * 0.72)
    aligned = [chunk for score, chunk in scored if score >= relative_floor][:4]
    if not aligned:
        aligned = [scored[0][1]]

    generation_context_parts = []
    for chunk in aligned:
        if chunk.get("parent_chunk_id") in parent_by_id:
            generation_context_parts.append(
                parent_by_id[chunk["parent_chunk_id"]]["content"]
            )
        else:
            generation_context_parts.append(chunk["content"])

    generation_context = "\n\n---\n\n".join(
        dict.fromkeys(generation_context_parts)
    )[:16000]

    evidence_linkage[seed_id] = {
        "label_basis": linkage_basis,
        "registered_document_ids": provenance["document_ids"],
        "positive_chunk_ids": [chunk["chunk_id"] for chunk in aligned],
        "positive_parent_chunk_ids": sorted({
            chunk.get("parent_chunk_id")
            for chunk in aligned
            if chunk.get("parent_chunk_id")
        }),
        "best_evidence_coverage": best_score,
        "generation_context": generation_context,
    }

weak_linkages = [
    seed_id for seed_id, item in evidence_linkage.items()
    if item["best_evidence_coverage"] < 0.18
]

print("Child chunks:", len(child_chunks))
print("Parent chunks:", len(parent_chunks))
print("Seeds linked:", len(evidence_linkage))
print("Weak evidence-to-chunk links (<0.18):", weak_linkages)
print("\nEVIDENCE-TO-CHUNK LINKAGE: PASS")


Child chunks: 2721
Parent chunks: 1447
Seeds linked: 28
Weak evidence-to-chunk links (<0.18): [22]

EVIDENCE-TO-CHUNK LINKAGE: PASS


<div style="direction: rtl; line-height: 1.8;">
<h2>۵. split بدون leakage</h2>
<p>
تمام سؤال‌های یک seed فقط در یک split قرار می‌گیرند. این کار قبل از تولید سؤال
انجام می‌شود؛ بنابراین نسخه‌های مختلف یک evidence بین train/dev/test پخش نمی‌شوند.
</p>
</div>


In [8]:
# Code Cell 08
def deterministic_split(seed_id: int) -> str:
    digest = hashlib.sha256(
        f"nb12g-split-v1:{seed_id}".encode("utf-8")
    ).digest()
    bucket = int.from_bytes(digest[:4], "big") % 100
    if bucket < 60:
        return "train"
    if bucket < 80:
        return "dev"
    return "test"


split_by_seed = {
    seed_id: deterministic_split(seed_id)
    for seed_id in seed_registry
}
split_counts = Counter(split_by_seed.values())
assert set(split_counts) == {"train", "dev", "test"}, split_counts

print("Seed-level split:")
for split_name in ("train", "dev", "test"):
    ids = [seed_id for seed_id, split in split_by_seed.items() if split == split_name]
    print(f"- {split_name}: {len(ids)} seeds -> {ids}")
print("\nLEAKAGE-SAFE SPLIT: PASS")


Seed-level split:
- train: 19 seeds -> [2, 3, 4, 6, 8, 9, 11, 13, 14, 15, 16, 17, 18, 19, 20, 22, 24, 26, 28]
- dev: 4 seeds -> [7, 12, 21, 25]
- test: 5 seeds -> [1, 5, 10, 23, 27]

LEAKAGE-SAFE SPLIT: PASS


<div style="direction: rtl; line-height: 1.8;">
<h2>۶. تولید resumable سؤال و پاسخ از evidence</h2>
<p>
هر seed در یک درخواست مستقل پردازش و فوراً cache می‌شود. قطع اینترنت یا توقف kernel
باعث از دست رفتن درخواست‌های موفق نمی‌شود. مدل اجازه ندارد از سؤال‌های v1 یا قالب‌های
عمومی استفاده کند.
</p>
</div>


In [9]:
# Code Cell 09
assert importlib.util.find_spec("openai") is not None, (
    "Install the openai package before running this notebook."
)
assert importlib.util.find_spec("dotenv") is not None, (
    "Install python-dotenv before running this notebook."
)

from dotenv import load_dotenv
from openai import OpenAI

env_path = project_root / ".env"
assert env_path.is_file(), f"Environment file is missing: {env_path}"
load_dotenv(env_path, override=False)

arvan_api_key = os.getenv("ARVAN_AI_API_KEY")
arvan_chat_url = os.getenv("ARVAN_CHAT_URL")
arvan_chat_model = os.getenv("ARVAN_CHAT_MODEL")
arvan_embedding_url = os.getenv("ARVAN_EMBEDDING_URL")

assert arvan_api_key, "ARVAN_AI_API_KEY is missing."
assert arvan_chat_url, "ARVAN_CHAT_URL is missing."
assert arvan_chat_model, "ARVAN_CHAT_MODEL is missing."
assert arvan_embedding_url, "ARVAN_EMBEDDING_URL is missing."
assert urlparse(arvan_chat_url).scheme == "https"
assert urlparse(arvan_embedding_url).scheme == "https"

chat_client = OpenAI(
    api_key=arvan_api_key,
    base_url=arvan_chat_url,
    timeout=180.0,
    max_retries=1,
)
embedding_client = OpenAI(
    api_key=arvan_api_key,
    base_url=arvan_embedding_url,
    timeout=90.0,
    max_retries=1,
)
embedding_model_id = "text-embedding-3-large"

build_identity = {
    "source_dataset_sha256": source_dataset_sha256,
    "freeze_manifest_sha256": sha256_file(freeze_manifest_path),
    "generation_prompt_version": GENERATION_PROMPT_VERSION,
    "judge_prompt_version": JUDGE_PROMPT_VERSION,
    "chat_model": arvan_chat_model,
    "questions_per_seed": QUESTIONS_PER_SEED,
}
build_fingerprint = hashlib.sha256(
    json.dumps(build_identity, sort_keys=True).encode("utf-8")
).hexdigest()

build_directory = (
    project_root / "artifacts" / "phase2" / "golden_dataset_rebuild"
    / build_fingerprint
)
generation_cache_directory = build_directory / "generation_cache"
judge_cache_directory = build_directory / "judge_cache"
embedding_cache_directory = build_directory / "embedding_cache"
for directory in (
    generation_cache_directory,
    judge_cache_directory,
    embedding_cache_directory,
):
    directory.mkdir(parents=True, exist_ok=True)

print("Chat model:", arvan_chat_model)
print("Embedding model:", embedding_model_id)
print("Build fingerprint:", build_fingerprint)
print("Build directory:", build_directory)
print("\nMODEL AND CACHE PREFLIGHT: PASS")


Chat model: Qwen3-30B-A3B
Embedding model: text-embedding-3-large
Build fingerprint: 273f5fe5eb52607c4bc8daa3a6dc92a85f29d90a8c40671048c8b1a3488b8690
Build directory: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/golden_dataset_rebuild/273f5fe5eb52607c4bc8daa3a6dc92a85f29d90a8c40671048c8b1a3488b8690

MODEL AND CACHE PREFLIGHT: PASS


In [10]:
# Code Cell 10
intent_contract = [
    "DIRECT_NATURAL",
    "SCENARIO_APPLICATION",
    "MISCONCEPTION_CORRECTION",
    "CAUSE_OR_COMPARISON",
]

forbidden_phrases = [
    "طبق متن", "بر اساس متن", "در این متن", "در این بخش", "منبع",
    "چه نکته ای باید بررسی شود", "چه مفهومی باید درک شود",
    "چه چیزی اهمیت بیشتری دارد", "برای تحلیل بهتر این موضوع",
]

def generation_prompt(seed_id: int) -> str:
    seed = seed_registry[seed_id]
    linkage = evidence_linkage[seed_id]
    return f"""
شما طراح ارشد Golden Dataset برای بازیابی اطلاعات فارسی هستید.
فقط از شواهد زیر استفاده کنید و هیچ دانش بیرونی اضافه نکنید.

هدف: دقیقاً {QUESTIONS_PER_SEED} پرسش طبیعی، مستقل و واقعاً متفاوت بسازید.
intentها دقیقاً و به همین ترتیب باشند:
{json.dumps(intent_contract, ensure_ascii=False)}

قواعد سخت:
- سؤال باید بدون دیدن منبع قابل فهم و self-contained باشد.
- از عبارت‌هایی مثل «طبق متن»، «در این بخش»، «منبع» و قالب‌های عمومی استفاده نکنید.
- سؤال باید حداقل دو نشانهٔ محتوایی متمایز از evidence داشته باشد.
- پاسخ کوتاه، دقیق و کاملاً قابل استنتاج از evidence باشد.
- برای هر مورد 1 یا 2 نقل‌قول کوتاه و عیناً موجود در SOURCE CONTEXT بدهید.
- هیچ سؤال یا پاسخی را تکرار نکنید.
- خروجی فقط JSON معتبر باشد؛ بدون markdown و توضیح بیرونی.

ساختار خروجی:
{{"items":[{{"intent":"DIRECT_NATURAL","question":"...؟","answer":"...","evidence_quotes":["..."]}}]}}

SEED ID: {seed_id}
TAXONOMY: {json.dumps(seed['taxonomy_topics'], ensure_ascii=False)}
COMPETENCIES: {json.dumps(seed['competency_ids'], ensure_ascii=False)}

HUMAN-APPROVED EVIDENCE SUMMARY:
{seed['approved_evidence']}

HUMAN-APPROVED ANSWER REFERENCE:
{seed['approved_answer']}

SOURCE CONTEXT:
{linkage['generation_context']}
""".strip()


def validate_generation_payload(payload, seed_id: int):
    errors = []
    if not isinstance(payload, dict):
        return ["payload_not_object"]
    items = payload.get("items")
    if not isinstance(items, list):
        return ["items_not_list"]
    if len(items) != QUESTIONS_PER_SEED:
        errors.append(f"item_count:{len(items)}")
    observed_intents = []
    for index, item in enumerate(items):
        if not isinstance(item, dict):
            errors.append(f"item_{index}:not_object")
            continue
        missing = {
            "intent", "question", "answer", "evidence_quotes"
        } - set(item)
        if missing:
            errors.append(f"item_{index}:missing:{sorted(missing)}")
            continue
        observed_intents.append(item["intent"])
        if not isinstance(item["evidence_quotes"], list) or not item["evidence_quotes"]:
            errors.append(f"item_{index}:invalid_quotes")
    if observed_intents != intent_contract:
        errors.append(f"intent_contract:{observed_intents}")
    return errors


print("Intent contract:", intent_contract)
print("Forbidden phrase count:", len(forbidden_phrases))
print("\nGENERATION CONTRACT: PASS")


Intent contract: ['DIRECT_NATURAL', 'SCENARIO_APPLICATION', 'MISCONCEPTION_CORRECTION', 'CAUSE_OR_COMPARISON']
Forbidden phrase count: 9

GENERATION CONTRACT: PASS


In [11]:
# Code Cell 11
def request_json(prompt: str, max_tokens: int = 4096):
    response = chat_client.chat.completions.create(
        model=arvan_chat_model,
        messages=[
            {
                "role": "system",
                "content": (
                    "Return only a valid JSON object. Do not use markdown. "
                    "All factual content must be grounded in the supplied evidence."
                ),
            },
            {"role": "user", "content": prompt},
        ],
        temperature=0.15,
        max_tokens=max_tokens,
    )
    assert response.choices, "Chat endpoint returned no choices."
    choice = response.choices[0]
    raw = (choice.message.content or "").strip()
    usage = getattr(response, "usage", None)
    return {
        "raw": raw,
        "parsed": extract_first_json_object(raw),
        "finish_reason": choice.finish_reason,
        "usage": (
            usage.model_dump() if usage is not None and hasattr(usage, "model_dump")
            else str(usage)
        ),
    }


generation_results = {}
print("=== RESUMABLE EVIDENCE-GROUNDED GENERATION ===")

for position, seed_id in enumerate(sorted(seed_registry), start=1):
    cache_path = generation_cache_directory / f"seed_{seed_id:02d}.json"
    cached = read_json(cache_path) if cache_path.is_file() else None
    cached_errors = (
        validate_generation_payload(cached.get("payload"), seed_id)
        if isinstance(cached, dict) else ["no_cache"]
    )

    if cached is not None and not cached_errors:
        generation_results[seed_id] = cached
        print(f"Seed {position}/28 | seed={seed_id} | cached | PASS")
        continue

    prompt = generation_prompt(seed_id)
    last_record = None
    for attempt in range(1, CHAT_MAX_ATTEMPTS + 1):
        response_record = request_json(prompt)
        errors = validate_generation_payload(response_record["parsed"], seed_id)
        last_record = {
            "seed_id": seed_id,
            "prompt_version": GENERATION_PROMPT_VERSION,
            "model": arvan_chat_model,
            "attempt": attempt,
            "payload": response_record["parsed"],
            "raw_response": response_record["raw"],
            "finish_reason": response_record["finish_reason"],
            "usage": response_record["usage"],
            "contract_errors": errors,
        }
        write_json_atomic(cache_path, last_record)
        if not errors:
            break
        prompt += (
            "\n\nپاسخ قبلی این خطاهای قراردادی را داشت: "
            + json.dumps(errors, ensure_ascii=False)
            + "\nکل JSON را از نو و صحیح برگردان."
        )
        time.sleep(0.2)

    assert last_record is not None and not last_record["contract_errors"], (
        f"Seed {seed_id} generation failed after {CHAT_MAX_ATTEMPTS} attempts: "
        f"{last_record['contract_errors'] if last_record else 'no response'}"
    )
    generation_results[seed_id] = last_record
    print(
        f"Seed {position}/28 | seed={seed_id} | "
        f"attempt={last_record['attempt']} | PASS"
    )

assert len(generation_results) == 28
print("\nEVIDENCE-GROUNDED GENERATION: PASS")


=== RESUMABLE EVIDENCE-GROUNDED GENERATION ===
Seed 1/28 | seed=1 | cached | PASS
Seed 2/28 | seed=2 | cached | PASS
Seed 3/28 | seed=3 | cached | PASS
Seed 4/28 | seed=4 | cached | PASS
Seed 5/28 | seed=5 | cached | PASS
Seed 6/28 | seed=6 | cached | PASS
Seed 7/28 | seed=7 | cached | PASS
Seed 8/28 | seed=8 | cached | PASS
Seed 9/28 | seed=9 | cached | PASS
Seed 10/28 | seed=10 | cached | PASS
Seed 11/28 | seed=11 | cached | PASS
Seed 12/28 | seed=12 | cached | PASS
Seed 13/28 | seed=13 | cached | PASS
Seed 14/28 | seed=14 | cached | PASS
Seed 15/28 | seed=15 | cached | PASS
Seed 16/28 | seed=16 | cached | PASS
Seed 17/28 | seed=17 | cached | PASS
Seed 18/28 | seed=18 | cached | PASS
Seed 19/28 | seed=19 | cached | PASS
Seed 20/28 | seed=20 | cached | PASS
Seed 21/28 | seed=21 | cached | PASS
Seed 22/28 | seed=22 | cached | PASS
Seed 23/28 | seed=23 | cached | PASS
Seed 24/28 | seed=24 | cached | PASS
Seed 25/28 | seed=25 | cached | PASS
Seed 26/28 | seed=26 | cached | PASS
Seed 27/2

<div style="direction: rtl; line-height: 1.8;">
<h2>۷. gateهای قطعی کیفیت</h2>
<p>
این gateها ساختاری نیستند: سؤال generic، اشارهٔ مبهم، شاهد غیرواقعی، شباهت شدید
بین سؤال‌ها و پاسخ فاقد پشتوانه را علامت می‌زنند. هیچ رکوردی صرفاً برای رسیدن به
عدد هدف پذیرفته نمی‌شود.
</p>
</div>


In [12]:
# Code Cell 12
all_chunk_documents = [canonical_text(chunk["content"]) for chunk in child_chunks]
document_frequency = Counter()
for document in all_chunk_documents:
    document_frequency.update(set(content_token_set(document)))
chunk_document_count = len(all_chunk_documents)

def token_idf(token: str) -> float:
    return math.log((1 + chunk_document_count) / (1 + document_frequency[token])) + 1.0


idf_distribution = np.asarray([
    token_idf(token) for token in document_frequency
])
high_idf_threshold = float(np.quantile(idf_distribution, 0.75))

def deterministic_record_validation(seed_id: int, item: dict):
    seed = seed_registry[seed_id]
    context = evidence_linkage[seed_id]["generation_context"]
    question = str(item.get("question", "")).strip()
    answer = str(item.get("answer", "")).strip()
    quotes = item.get("evidence_quotes", [])
    failures = []

    question_tokens = tokens(question)
    answer_tokens = content_token_set(answer)
    context_tokens = content_token_set(context)
    question_content = content_token_set(question)

    if not (6 <= len(question_tokens) <= 45):
        failures.append("question_length")
    if not question.endswith("؟"):
        failures.append("missing_persian_question_mark")
    if any(phrase in canonical_text(question) for phrase in forbidden_phrases):
        failures.append("generic_or_deictic_phrase")
    if len(question_content & context_tokens) < 2:
        failures.append("insufficient_question_evidence_signal")
    if not (8 <= len(tokens(answer)) <= 180):
        failures.append("answer_length")
    answer_support = (
        len(answer_tokens & context_tokens) / len(answer_tokens)
        if answer_tokens else 0.0
    )
    if answer_support < 0.20:
        failures.append("weak_answer_lexical_support")

    canonical_context = canonical_text(context)
    valid_quotes = [
        quote for quote in quotes
        if isinstance(quote, str)
        and 2 <= len(tokens(quote)) <= 35
        and canonical_text(quote) in canonical_context
    ]
    if len(valid_quotes) != len(quotes) or not valid_quotes:
        failures.append("non_exact_or_invalid_evidence_quote")

    high_idf_tokens = sorted({
        token for token in question_content
        if token_idf(token) >= high_idf_threshold
    })
    if not high_idf_tokens:
        failures.append("no_high_idf_query_signal")

    return {
        "failures": sorted(set(failures)),
        "answer_lexical_support": answer_support,
        "question_context_shared_tokens": len(question_content & context_tokens),
        "high_idf_tokens": high_idf_tokens,
        "valid_evidence_quotes": valid_quotes,
    }


generated_rows = []
for seed_id, result in sorted(generation_results.items()):
    for item_index, item in enumerate(result["payload"]["items"], start=1):
        validation = deterministic_record_validation(seed_id, item)
        generated_rows.append({
            "source_seed_id": seed_id,
            "item_index": item_index,
            "intent": item["intent"],
            "question": item["question"].strip(),
            "expected_answer": item["answer"].strip(),
            "evidence_quotes": validation["valid_evidence_quotes"],
            "deterministic_validation": validation,
        })

canonical_question_counts = Counter(
    canonical_text(row["question"]) for row in generated_rows
)
for row in generated_rows:
    if canonical_question_counts[canonical_text(row["question"])] > 1:
        row["deterministic_validation"]["failures"].append(
            "duplicate_canonical_question"
        )

assert importlib.util.find_spec("sklearn") is not None
from sklearn.feature_extraction.text import TfidfVectorizer

near_duplicate_pairs = []
for seed_id in seed_registry:
    rows = [row for row in generated_rows if row["source_seed_id"] == seed_id]
    matrix = TfidfVectorizer(
        analyzer="char_wb", ngram_range=(3, 5), min_df=1
    ).fit_transform([canonical_text(row["question"]) for row in rows])
    similarities = (matrix @ matrix.T).toarray()
    for left in range(len(rows)):
        for right in range(left + 1, len(rows)):
            if similarities[left, right] >= 0.88:
                near_duplicate_pairs.append({
                    "seed_id": seed_id,
                    "left_item": rows[left]["item_index"],
                    "right_item": rows[right]["item_index"],
                    "similarity": float(similarities[left, right]),
                })
                rows[left]["deterministic_validation"]["failures"].append(
                    "near_duplicate_within_seed"
                )
                rows[right]["deterministic_validation"]["failures"].append(
                    "near_duplicate_within_seed"
                )

deterministic_pass_count = sum(
    not row["deterministic_validation"]["failures"]
    for row in generated_rows
)
failure_counts = Counter(
    failure
    for row in generated_rows
    for failure in row["deterministic_validation"]["failures"]
)

print("Generated rows:", len(generated_rows))
print("Deterministic PASS:", deterministic_pass_count)
print("Deterministic REVIEW:", len(generated_rows) - deterministic_pass_count)
print("Failure counts:", dict(failure_counts))
print("Near-duplicate pairs:", len(near_duplicate_pairs))
print("High-IDF threshold:", round(high_idf_threshold, 4))
print("\nDETERMINISTIC QUALITY GATES: PASS")


Generated rows: 112
Deterministic PASS: 0
Deterministic REVIEW: 112
Failure counts: {'insufficient_question_evidence_signal': 3, 'no_high_idf_query_signal': 101, 'non_exact_or_invalid_evidence_quote': 92, 'weak_answer_lexical_support': 7, 'answer_length': 6, 'question_length': 1}
Near-duplicate pairs: 0
High-IDF threshold: 8.216

DETERMINISTIC QUALITY GATES: PASS


<div style="direction: rtl; line-height: 1.8;">
<h2>۸. داوری ماشینی مستقل از مرحلهٔ تولید</h2>
<p>
داوری در درخواست جداگانه و با rubric ثابت انجام می‌شود. این جایگزین قطعی نظر انسان
نیست، اما صف بازبینی را به موارد کم‌اطمینان محدود می‌کند.
</p>
</div>


In [13]:
# Code Cell 13
def judge_prompt(seed_id: int, rows: list[dict]) -> str:
    seed = seed_registry[seed_id]
    public_rows = [
        {
            "item_index": row["item_index"],
            "intent": row["intent"],
            "question": row["question"],
            "answer": row["expected_answer"],
            "evidence_quotes": row["evidence_quotes"],
        }
        for row in rows
    ]
    return f"""
نقش شما داور سخت‌گیر Golden Dataset است. فقط با evidence زیر داوری کنید.
برای هر item از 1 تا 5 به groundedness، self_containedness، naturalness و
specificity امتیاز دهید. accepted فقط وقتی true است که هر چهار امتیاز حداقل 4
باشند و سؤال با سؤال‌های دیگر intent تکراری نداشته باشد.
خروجی فقط JSON معتبر با این ساختار باشد:
{{"assessments":[{{"item_index":1,"groundedness":5,"self_containedness":5,
"naturalness":5,"specificity":5,"accepted":true,"reason":"کوتاه"}}]}}

EVIDENCE:
{seed['approved_evidence']}

SOURCE CONTEXT:
{evidence_linkage[seed_id]['generation_context']}

ITEMS:
{json.dumps(public_rows, ensure_ascii=False, indent=2)}
""".strip()


def validate_judge_payload(payload):
    if not isinstance(payload, dict) or not isinstance(payload.get("assessments"), list):
        return ["invalid_assessments"]
    assessments = payload["assessments"]
    if len(assessments) != QUESTIONS_PER_SEED:
        return [f"assessment_count:{len(assessments)}"]
    indexes = []
    errors = []
    for assessment in assessments:
        if not isinstance(assessment, dict):
            errors.append("assessment_not_object")
            continue
        indexes.append(assessment.get("item_index"))
        for field in (
            "groundedness", "self_containedness", "naturalness", "specificity"
        ):
            score = assessment.get(field)
            if not isinstance(score, int) or not 1 <= score <= 5:
                errors.append(f"invalid_{field}")
        if not isinstance(assessment.get("accepted"), bool):
            errors.append("invalid_accepted")
    if sorted(indexes) != list(range(1, QUESTIONS_PER_SEED + 1)):
        errors.append(f"invalid_indexes:{indexes}")
    return sorted(set(errors))


judge_results = {}
print("=== RESUMABLE MACHINE JUDGMENT ===")
for position, seed_id in enumerate(sorted(seed_registry), start=1):
    seed_rows = [row for row in generated_rows if row["source_seed_id"] == seed_id]
    cache_path = judge_cache_directory / f"seed_{seed_id:02d}.json"
    cached = read_json(cache_path) if cache_path.is_file() else None
    cached_errors = (
        validate_judge_payload(cached.get("payload"))
        if isinstance(cached, dict) else ["no_cache"]
    )
    if cached is not None and not cached_errors:
        judge_results[seed_id] = cached
        print(f"Seed {position}/28 | seed={seed_id} | cached | PASS")
        continue

    prompt = judge_prompt(seed_id, seed_rows)
    last_record = None
    for attempt in range(1, CHAT_MAX_ATTEMPTS + 1):
        response_record = request_json(prompt, max_tokens=3072)
        errors = validate_judge_payload(response_record["parsed"])
        last_record = {
            "seed_id": seed_id,
            "prompt_version": JUDGE_PROMPT_VERSION,
            "model": arvan_chat_model,
            "attempt": attempt,
            "payload": response_record["parsed"],
            "raw_response": response_record["raw"],
            "finish_reason": response_record["finish_reason"],
            "usage": response_record["usage"],
            "contract_errors": errors,
        }
        write_json_atomic(cache_path, last_record)
        if not errors:
            break
        prompt += "\n\nJSON قبلی نامعتبر بود. کل خروجی را صحیح بازسازی کن."
        time.sleep(0.2)
    assert last_record is not None and not last_record["contract_errors"], (
        f"Machine judge failed for seed {seed_id}: {last_record}"
    )
    judge_results[seed_id] = last_record
    print(f"Seed {position}/28 | seed={seed_id} | PASS")

assessment_by_key = {
    (seed_id, assessment["item_index"]): assessment
    for seed_id, result in judge_results.items()
    for assessment in result["payload"]["assessments"]
}
assert len(assessment_by_key) == len(generated_rows)
print("\nMACHINE JUDGMENT: PASS")


=== RESUMABLE MACHINE JUDGMENT ===
Seed 1/28 | seed=1 | cached | PASS
Seed 2/28 | seed=2 | cached | PASS
Seed 3/28 | seed=3 | cached | PASS
Seed 4/28 | seed=4 | cached | PASS
Seed 5/28 | seed=5 | cached | PASS
Seed 6/28 | seed=6 | cached | PASS
Seed 7/28 | seed=7 | cached | PASS
Seed 8/28 | seed=8 | cached | PASS
Seed 9/28 | seed=9 | cached | PASS
Seed 10/28 | seed=10 | cached | PASS
Seed 11/28 | seed=11 | cached | PASS
Seed 12/28 | seed=12 | cached | PASS
Seed 13/28 | seed=13 | cached | PASS
Seed 14/28 | seed=14 | cached | PASS
Seed 15/28 | seed=15 | cached | PASS
Seed 16/28 | seed=16 | cached | PASS
Seed 17/28 | seed=17 | cached | PASS
Seed 18/28 | seed=18 | cached | PASS
Seed 19/28 | seed=19 | cached | PASS
Seed 20/28 | seed=20 | cached | PASS
Seed 21/28 | seed=21 | cached | PASS
Seed 22/28 | seed=22 | cached | PASS
Seed 23/28 | seed=23 | cached | PASS
Seed 24/28 | seed=24 | cached | PASS
Seed 25/28 | seed=25 | cached | PASS
Seed 26/28 | seed=26 | cached | PASS
Seed 27/28 | seed=27 

<div style="direction: rtl; line-height: 1.8;">
<h2>۹. ساخت رکوردهای candidate و صف بازبینی هدفمند</h2>
<p>
همهٔ رکوردها provenance و labelهای retrieval دارند. موارد دارای شکست deterministic
یا امتیاز پایین داور وارد review queue می‌شوند. علاوه بر آن، یک نمونهٔ کنترل ثابت
از رکوردهای پاس‌شده برای audit انسانی انتخاب می‌شود.
</p>
</div>


In [14]:
# Code Cell 14
candidate_records = []
review_queue = []

for row in generated_rows:
    seed_id = row["source_seed_id"]
    seed = seed_registry[seed_id]
    provenance = provenance_registry[seed_id]
    linkage = evidence_linkage[seed_id]
    judge = assessment_by_key[(seed_id, row["item_index"])]
    deterministic_failures = sorted(set(
        row["deterministic_validation"]["failures"]
    ))
    machine_accepted = bool(judge["accepted"]) and all(
        judge[field] >= 4
        for field in (
            "groundedness", "self_containedness", "naturalness", "specificity"
        )
    )
    needs_review = bool(deterministic_failures) or not machine_accepted

    stable_name = (
        f"{build_fingerprint}:{seed_id}:{row['item_index']}:"
        f"{canonical_text(row['question'])}"
    )
    candidate_id = str(uuid.uuid5(uuid.NAMESPACE_URL, stable_name))

    record = {
        "candidate_id": candidate_id,
        "dataset_version": "golden_dataset_rebuild_experiment",
        "question": row["question"],
        "expected_answer": row["expected_answer"],
        "source_seed_id": seed_id,
        "source_lineage": seed["source_lineage"],
        "source_evidence": seed["approved_evidence"],
        "evidence_quotes": row["evidence_quotes"],
        "taxonomy_topics": seed["taxonomy_topics"],
        "competency_ids": seed["competency_ids"],
        "question_intent": row["intent"],
        "evaluation_split": split_by_seed[seed_id],
        "pilot_case_id": provenance["pilot_case_id"],
        "provenance_source": provenance["provenance_source"],
        "document_ids": provenance["document_ids"],
        "retrieval_labels": {
            "candidate": "parent_child_children",
            "positive_chunk_ids": linkage["positive_chunk_ids"],
            "positive_parent_chunk_ids": linkage["positive_parent_chunk_ids"],
            "label_basis": linkage["label_basis"],
        },
        "generation": {
            "model": arvan_chat_model,
            "prompt_version": GENERATION_PROMPT_VERSION,
            "build_fingerprint": build_fingerprint,
        },
        "quality": {
            "deterministic": row["deterministic_validation"],
            "machine_judge": judge,
            "requires_human_review": needs_review,
        },
        "generation_status": (
            "V2_MACHINE_ACCEPTED" if not needs_review else "V2_REVIEW_REQUIRED"
        ),
    }
    candidate_records.append(record)

    if needs_review:
        review_queue.append({
            "candidate_id": candidate_id,
            "source_seed_id": seed_id,
            "question": row["question"],
            "expected_answer": row["expected_answer"],
            "evidence_quotes": row["evidence_quotes"],
            "review_reasons": deterministic_failures + (
                [] if machine_accepted else ["machine_judge_rejected"]
            ),
            "machine_judge": judge,
            "human_decision": None,
            "human_notes": None,
        })

assert len(candidate_records) == 28 * QUESTIONS_PER_SEED
assert len({record["candidate_id"] for record in candidate_records}) == len(candidate_records)
assert len({canonical_text(record["question"]) for record in candidate_records}) == len(candidate_records)

# Deterministic 10% audit sample from machine-accepted records.
accepted_records = [
    record for record in candidate_records
    if not record["quality"]["requires_human_review"]
]
audit_size = max(1, math.ceil(len(accepted_records) * 0.10))
audit_sample = sorted(
    accepted_records,
    key=lambda record: hashlib.sha256(
        f"nb12g-audit:{record['candidate_id']}".encode("utf-8")
    ).hexdigest(),
)[:audit_size]

queued_ids = {item["candidate_id"] for item in review_queue}
for record in audit_sample:
    if record["candidate_id"] in queued_ids:
        continue
    review_queue.append({
        "candidate_id": record["candidate_id"],
        "source_seed_id": record["source_seed_id"],
        "question": record["question"],
        "expected_answer": record["expected_answer"],
        "evidence_quotes": record["evidence_quotes"],
        "review_reasons": ["random_quality_audit"],
        "machine_judge": record["quality"]["machine_judge"],
        "human_decision": None,
        "human_notes": None,
    })

experiment_path = golden_directory / f"{EXPERIMENT_DATASET_STEM}.jsonl"
review_queue_path = golden_directory / f"{EXPERIMENT_DATASET_STEM}_review_queue.jsonl"
write_jsonl_atomic(experiment_path, candidate_records)
write_jsonl_atomic(review_queue_path, review_queue)

print("Candidate records:", len(candidate_records))
print("Machine-review required:", sum(
    record["quality"]["requires_human_review"] for record in candidate_records
))
print("Human review queue including 10% audit:", len(review_queue))
print("Non-active experiment:", experiment_path)
print("Review queue:", review_queue_path)
print("\nV2 CANDIDATE ASSEMBLY: PASS")


Candidate records: 112
Machine-review required: 112
Human review queue including 10% audit: 112
Non-active experiment: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_rebuild_experiment.jsonl
Review queue: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_rebuild_experiment_review_queue.jsonl

V2 CANDIDATE ASSEMBLY: PASS


<div style="direction: rtl; line-height: 1.8;">
<h2>۱۰. benchmark تشخیصی مستقل (بدون سوگیری انتخاب)</h2>
<p>
ابتدا تمام سؤال‌ها ثابت می‌شوند، سپس retrieval اندازه‌گیری می‌شود. هیچ سؤال ضعیفی
به‌خاطر rank حذف نمی‌شود. cache embedding بر اساس hash متن و build fingerprint است؛
تغییر نام نوت‌بوک یا Restart باعث محاسبهٔ دوباره نمی‌شود.
</p>
</div>


In [15]:
# Code Cell 15
embedding_text_by_hash = {}
for record in candidate_records:
    embedding_text_by_hash[text_hash(record["question"])] = record["question"]
for chunk in child_chunks:
    embedding_text_by_hash[text_hash(chunk["content"])] = canonical_text(chunk["content"])

embedding_batch_directory = embedding_cache_directory / embedding_model_id / "batches"
embedding_batch_directory.mkdir(parents=True, exist_ok=True)

vectors_by_hash = {}
for batch_path in sorted(embedding_batch_directory.glob("batch_*.npz")):
    with np.load(batch_path, allow_pickle=False) as data:
        batch_hashes = [str(value) for value in data["text_hashes"]]
        batch_vectors = data["vectors"].astype(np.float32)
    assert len(batch_hashes) == batch_vectors.shape[0]
    vectors_by_hash.update(dict(zip(batch_hashes, batch_vectors)))

# Reuse compatible NB13 Arvan batch caches when present.
for batch_path in sorted(
    (project_root / "artifacts" / "phase2" / "embedding_benchmark").glob(
        "*/arvan_text_embedding_3_large/batches/batch_*.npz"
    )
):
    try:
        with np.load(batch_path, allow_pickle=False) as data:
            if "input_hashes" not in data or "embeddings" not in data:
                continue
            old_hashes = [str(value) for value in data["input_hashes"]]
            old_vectors = data["embeddings"].astype(np.float32)
        for old_hash, vector in zip(old_hashes, old_vectors):
            if old_hash in embedding_text_by_hash and old_hash not in vectors_by_hash:
                vectors_by_hash[old_hash] = vector
    except Exception:
        continue

pending_hashes = sorted(set(embedding_text_by_hash) - set(vectors_by_hash))
batches = [pending_hashes[index:index + 64] for index in range(0, len(pending_hashes), 64)]

print("=== RESUMABLE EMBEDDING CACHE ===")
print("Texts required:", len(embedding_text_by_hash))
print("Vectors reused:", len(vectors_by_hash))
print("Texts pending:", len(pending_hashes))

for batch_number, batch_hashes in enumerate(batches, start=1):
    response = embedding_client.embeddings.create(
        model=embedding_model_id,
        input=[embedding_text_by_hash[item] for item in batch_hashes],
    )
    response_items = sorted(response.data, key=lambda item: item.index)
    batch_vectors = np.asarray(
        [item.embedding for item in response_items], dtype=np.float32
    )
    assert batch_vectors.shape[0] == len(batch_hashes)
    assert not np.isnan(batch_vectors).any()
    norms = np.linalg.norm(batch_vectors, axis=1, keepdims=True)
    batch_vectors = batch_vectors / np.maximum(norms, 1e-12)
    batch_identity = hashlib.sha256(
        "|".join(batch_hashes).encode("utf-8")
    ).hexdigest()[:16]
    output_path = embedding_batch_directory / f"batch_{batch_identity}.npz"
    np.savez_compressed(
        output_path,
        text_hashes=np.asarray(batch_hashes),
        vectors=batch_vectors,
    )
    vectors_by_hash.update(dict(zip(batch_hashes, batch_vectors)))
    print(f"Batch {batch_number}/{len(batches)} saved")
    time.sleep(0.1)

assert set(embedding_text_by_hash).issubset(vectors_by_hash)
print("\nRESUMABLE EMBEDDING CACHE: PASS")


=== RESUMABLE EMBEDDING CACHE ===
Texts required: 2821
Vectors reused: 2821
Texts pending: 0

RESUMABLE EMBEDDING CACHE: PASS


In [16]:
# Code Cell 16
child_ids = [chunk["chunk_id"] for chunk in child_chunks]
child_texts = [canonical_text(chunk["content"]) for chunk in child_chunks]
passage_matrix = np.vstack([
    vectors_by_hash[text_hash(text)] for text in child_texts
]).astype(np.float32)
passage_matrix /= np.maximum(
    np.linalg.norm(passage_matrix, axis=1, keepdims=True), 1e-12
)

lexical_vectorizer = TfidfVectorizer(
    tokenizer=tokens,
    token_pattern=None,
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True,
    norm="l2",
)
lexical_passage_matrix = lexical_vectorizer.fit_transform(child_texts)

def rank_positions_descending(scores: np.ndarray) -> np.ndarray:
    order = np.argsort(-scores, kind="stable")
    positions = np.empty(len(order), dtype=np.int32)
    positions[order] = np.arange(1, len(order) + 1, dtype=np.int32)
    return positions


benchmark_rows = []
for record in candidate_records:
    query = record["question"]
    query_vector = vectors_by_hash[text_hash(query)].astype(np.float32)
    query_vector /= max(float(np.linalg.norm(query_vector)), 1e-12)
    dense_scores = passage_matrix @ query_vector

    lexical_query = lexical_vectorizer.transform([canonical_text(query)])
    lexical_scores = (lexical_passage_matrix @ lexical_query.T).toarray().ravel()

    dense_ranks = rank_positions_descending(dense_scores)
    lexical_ranks = rank_positions_descending(lexical_scores)
    rrf_scores = 1.0 / (60 + dense_ranks) + 1.0 / (60 + lexical_ranks)
    final_order = np.argsort(-rrf_scores, kind="stable")

    positives = set(record["retrieval_labels"]["positive_chunk_ids"])
    positive_indexes = [
        index for index, chunk_id in enumerate(child_ids) if chunk_id in positives
    ]
    if positive_indexes:
        final_positions = np.empty(len(final_order), dtype=np.int32)
        final_positions[final_order] = np.arange(1, len(final_order) + 1)
        first_positive_rank = int(min(final_positions[index] for index in positive_indexes))
    else:
        first_positive_rank = None

    benchmark_rows.append({
        "candidate_id": record["candidate_id"],
        "source_seed_id": record["source_seed_id"],
        "evaluation_split": record["evaluation_split"],
        "first_positive_rank": first_positive_rank,
        "hit_at_1": first_positive_rank is not None and first_positive_rank <= 1,
        "hit_at_3": first_positive_rank is not None and first_positive_rank <= 3,
        "hit_at_5": first_positive_rank is not None and first_positive_rank <= 5,
        "hit_at_10": first_positive_rank is not None and first_positive_rank <= 10,
        "reciprocal_rank": (
            1.0 / first_positive_rank if first_positive_rank else 0.0
        ),
    })

eligible_benchmark_rows = [
    row for row in benchmark_rows if row["first_positive_rank"] is not None
]
assert eligible_benchmark_rows, "No records have usable retrieval labels."

retrieval_metrics = {
    "eligible_queries": len(eligible_benchmark_rows),
    "recall_at_1": float(np.mean([row["hit_at_1"] for row in eligible_benchmark_rows])),
    "recall_at_3": float(np.mean([row["hit_at_3"] for row in eligible_benchmark_rows])),
    "recall_at_5": float(np.mean([row["hit_at_5"] for row in eligible_benchmark_rows])),
    "recall_at_10": float(np.mean([row["hit_at_10"] for row in eligible_benchmark_rows])),
    "mrr": float(np.mean([row["reciprocal_rank"] for row in eligible_benchmark_rows])),
    "median_first_positive_rank": float(np.median([
        row["first_positive_rank"] for row in eligible_benchmark_rows
    ])),
}

print("=== V2 CANDIDATE RETRIEVAL DIAGNOSTIC ===")
for key, value in retrieval_metrics.items():
    print(f"- {key}: {value}")
print("Important: retrieval rank was not used to accept or reject records.")
print("\nV2 RETRIEVAL DIAGNOSTIC: PASS")


=== V2 CANDIDATE RETRIEVAL DIAGNOSTIC ===
- eligible_queries: 108
- recall_at_1: 0.3148148148148148
- recall_at_3: 0.4722222222222222
- recall_at_5: 0.5277777777777778
- recall_at_10: 0.6203703703703703
- mrr: 0.4207269974457064
- median_first_positive_rank: 4.5
Important: retrieval rank was not used to accept or reject records.

V2 RETRIEVAL DIAGNOSTIC: PASS


<div style="direction: rtl; line-height: 1.8;">
<h2>۱۱. گزارش نهایی آزمایش</h2>
<p>
گزارش hash واقعی را بعد از نوشتن فایل محاسبه می‌کند؛ هیچ SHA-256 hard-code نمی‌شود.
این گزارش صرفاً تصمیم‌گیری برای بازسازی آینده را پشتیبانی می‌کند و production approval نمی‌دهد.
</p>
</div>


In [17]:
# Code Cell 17
split_record_counts = Counter(
    record["evaluation_split"] for record in candidate_records
)
intent_counts = Counter(
    record["question_intent"] for record in candidate_records
)
review_required_count = sum(
    record["quality"]["requires_human_review"]
    for record in candidate_records
)

quality_report = {
    "schema_version": "golden_dataset_rebuild_experiment_report_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "build_fingerprint": build_fingerprint,
    "source": {
        "dataset": str(source_dataset_path.relative_to(project_root)),
        "sha256": source_dataset_sha256,
        "records": len(source_records),
        "questions_used_for_generation": False,
    },
    "experiment": {
        "path": str(experiment_path.relative_to(project_root)),
        "sha256": sha256_file(experiment_path),
        "records": len(candidate_records),
        "seeds": len(seed_registry),
        "questions_per_seed": QUESTIONS_PER_SEED,
        "split_records": dict(split_record_counts),
        "intent_counts": dict(intent_counts),
    },
    "quality": {
        "deterministic_pass": deterministic_pass_count,
        "deterministic_failure_counts": dict(failure_counts),
        "machine_review_required": review_required_count,
        "human_review_queue_records": len(review_queue),
        "near_duplicate_pairs": near_duplicate_pairs,
    },
    "retrieval_diagnostic": {
        "configuration": "arvan_text_embedding_3_large_plus_persian_tfidf_rrf",
        "candidate": "parent_child_children",
        "metrics": retrieval_metrics,
        "selection_bias_guard": "RANK_NOT_USED_AS_ACCEPTANCE_GATE",
    },
    "status": "EXPERIMENT_COMPLETE_NOT_A_GOLDEN_VERSION",
    "production_approved": False,
}

quality_report_path = golden_directory / f"{EXPERIMENT_DATASET_STEM}_quality_report.json"
benchmark_report_path = golden_directory / f"{EXPERIMENT_DATASET_STEM}_retrieval_diagnostic.json"
build_manifest_path = golden_directory / f"{EXPERIMENT_DATASET_STEM}_build_manifest.json"

write_json_atomic(quality_report_path, quality_report)
write_json_atomic(benchmark_report_path, {
    "build_fingerprint": build_fingerprint,
    "metrics": retrieval_metrics,
    "rows": benchmark_rows,
})
write_json_atomic(build_manifest_path, {
    **build_identity,
    "build_fingerprint": build_fingerprint,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "experiment_path": str(experiment_path.relative_to(project_root)),
    "experiment_sha256": sha256_file(experiment_path),
    "review_queue_path": str(review_queue_path.relative_to(project_root)),
    "review_queue_sha256": sha256_file(review_queue_path),
    "creates_active_dataset": False,
})

print("=== GOLDEN DATASET REBUILD EXPERIMENT REPORT ===")
print("Status:", quality_report["status"])
print("Records:", len(candidate_records))
print("Review queue:", len(review_queue))
print("Retrieval Recall@10:", f"{retrieval_metrics['recall_at_10']:.4f}")
print("Retrieval MRR:", f"{retrieval_metrics['mrr']:.4f}")
print("Experiment SHA-256:", sha256_file(experiment_path))
print("Quality report:", quality_report_path)
print("\nV2 CANDIDATE REPORT: PASS")


=== GOLDEN DATASET REBUILD EXPERIMENT REPORT ===
Status: EXPERIMENT_COMPLETE_NOT_A_GOLDEN_VERSION
Records: 112
Review queue: 112
Retrieval Recall@10: 0.6204
Retrieval MRR: 0.4207
Experiment SHA-256: 245fb62db34ea739c8a643876d79a9136791a91aa5d049e50f131fc3ddaeade4
Quality report: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_rebuild_experiment_quality_report.json

V2 CANDIDATE REPORT: PASS


<div style="direction: rtl; line-height: 1.8;">
<h2>۱۲. مرز تحویل آزمایش</h2>
<p>
این نوت‌بوک عمداً هیچ promotionی انجام نمی‌دهد. فایل آزمایشی نام نسخه‌ای ندارد،
بنابراین loader فعلی NB13 آن را انتخاب نمی‌کند و Golden Dataset v1 دست‌نخورده می‌ماند.
</p>
</div>


In [18]:
# Code Cell 18
print("Experiment dataset:", experiment_path)
print("Active v1 remains:", source_dataset_path)
print("NB13 active loader will continue to select v1.")
print("\nEXPERIMENT HANDOFF BOUNDARY: PASS")


Experiment dataset: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_rebuild_experiment.jsonl
Active v1 remains: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/golden_dataset/golden_dataset_v1.jsonl
NB13 active loader will continue to select v1.

EXPERIMENT HANDOFF BOUNDARY: PASS


<div style="direction: rtl; line-height: 1.8;">
<h2>نتیجه و ترتیب اجرا</h2>
<ol>
  <li>NB10 و NB11 لازم نیست دوباره نوشته شوند؛ فقط artifactهای freezeشدهٔ آن‌ها باید موجود باشند.</li>
  <li>این نوت‌بوک را در <code>notebooks/phase2/</code> قرار دهید و Restart + Run All بزنید.</li>
  <li>خروجی در <code>golden_dataset_rebuild_experiment.jsonl</code> ساخته می‌شود و active نیست.</li>
  <li>پس از تحلیل آزمایش و بازخورد دکتر، جریان جداگانهٔ ساخت v2 طراحی و اجرا می‌شود.</li>
</ol>
<p>
این آزمایش هیچ نسخه‌ای را منتشر یا فعال نمی‌کند؛ هدفش فقط اثبات یا رد مسیر بازسازی است.
</p>
</div>
